# Omitted and added triples analysis

This notebook calculates omitted and added triple slots for the GEM 2024 content ordering and structuring outputs.

Only these datasets are evaluated:

- factual
- fictional
- counterfactual

The model outputs for ordering and structuring are predicate sequences, not full triples. The notebook therefore treats each predicted predicate as one predicted triple slot. A predicted predicate is matched to one source triple with the same predicate, following the same consumption logic used in `mapping.py`. Source triples left unmatched are counted as omitted. Predicted predicates left unmatched are counted as added.

Comparison baselines:

- ordering is checked against the original WebNLG-17 input stored in each ordering result JSON record.
- structuring is checked against its direct input, which is the ordering output.
- structuring is also checked against the original WebNLG-17 input by aligning with the ordering result JSON for the same dataset and `idx`.

Important id columns:

- `triple_set_id` is the source example id from the result JSON, usually the same as `idx`.
- `source_triple_index` is the zero-based position of an omitted triple inside that triple set.
- `predicted_predicate_index` is the zero-based position of an added predicate inside the generated output.

In [72]:
from pathlib import Path
import csv
import json
import re
from collections import defaultdict

try:
    import pandas as pd
except ImportError:
    pd = None

PROJECT_DIR = Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUT_DIR = RESULTS_DIR / "omitted_triples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["factual", "fictional", "counterfactual"]

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"Pandas available:   {pd is not None}")

Project directory: /Users/chinonsoosuji/Python_projects/PHD PROJECTS/GEM2024_ST
Output directory:  /Users/chinonsoosuji/Python_projects/PHD PROJECTS/GEM2024_ST/results/omitted_triples
Pandas available:   True


## Helper functions

In [73]:
TRIPLE_RE = re.compile(r"\[TRIPLE\](.*?)\[/TRIPLE\]", flags=re.IGNORECASE | re.DOTALL)
TAG_RE = re.compile(r"\[/?(?:SNT|TRIPLE)\]", flags=re.IGNORECASE)


def normalise_tags(text):
    """Convert angle-bracket tags (<TRIPLE>) to square-bracket form ([TRIPLE]) and collapse whitespace."""
    text = str(text).replace("<", "[").replace(">", "]")
    return re.sub(r"\s+", " ", text).strip()


def parse_triples(text):
    """
    Extract source triples from text that uses [TRIPLE]...[/TRIPLE] tags.

    Each match is split into three parts by whitespace (max split = 2):
      - parts[0] → subject
      - parts[1] → predicate
      - parts[2] → object (everything after the second space; empty if missing)

    Returns a list of dicts, one per triple, with a zero-based source_triple_index
    that identifies each triple's position inside the triple set.
    """
    text = normalise_tags(text)
    triples = []

    for triple_index, match in enumerate(TRIPLE_RE.finditer(text)):
        raw = re.sub(r"\s+", " ", match.group(1)).strip()
        parts = raw.split(maxsplit=2)
        if len(parts) >= 2:
            triples.append({
                "source_triple_index": triple_index,
                "subject": parts[0],
                "predicate": parts[1],
                "object": parts[2] if len(parts) == 3 else "",
                "raw": raw,
            })

    return triples


def clean_predicate_token(token):
    """Strip surrounding punctuation and bracket characters from a single token."""
    token = token.strip().strip(",.;:(){}\"'")
    token = token.replace("[", "").replace("]", "")
    return token.strip()


def parse_predicted_predicates(text):
    """
    Extract the sequence of predicates from a model's ordering or structuring output.

    The ordering and structuring models output a flat sequence of predicate tokens
    (not full subject-predicate-object triples), so we:
      1. Remove all [TRIPLE] / [SNT] tags by replacing them with spaces.
      2. Treat every remaining whitespace-delimited token as one predicted predicate.

    Returns a list of predicate strings in output order.
    """
    text = normalise_tags(text)
    text = TAG_RE.sub(" ", text)
    predicates = []

    for token in text.split():
        token = clean_predicate_token(token)
        if not token:
            continue
        # Safety guard: skip any tag residue that wasn't caught by TAG_RE
        if token.upper() in {"SNT", "/SNT", "TRIPLE", "/TRIPLE"}:
            continue
        predicates.append(token)

    return predicates


def compare_predicates_to_source(source_text, pred_text):
    """
    Match predicted predicates to source triples and return omissions and additions.

    Matching is GREEDY and CASE-SENSITIVE:
      - We scan predicted predicates in output order.
      - Each predicted predicate is paired with the first *unmatched* source triple
        whose predicate field is an exact string match (case matters).
      - Once a source triple or predicted predicate is matched it cannot be used again.

    Why case-sensitive?
      A model predicting 'ICAOLocationIdentifier' when the source says
      'icaoLocationIdentifier' made a real error — wrong capitalisation means
      the predicate name is technically wrong. Such a case is counted as one
      OMISSION (source triple unmatched) AND one ADDITION (predicted predicate
      unmatched). This is intentional and correct for this evaluation.

    Returns:
        source_triples            — all parsed source triples
        predicted_predicates      — all parsed predicted predicates
        matched_source_indices    — set of source indices that were matched
        matched_predicate_indices — set of predicted indices that were matched
        omitted  — source triples with no matching predicted predicate (OMISSIONS)
        added    — predicted predicates with no matching source triple  (ADDITIONS)
    """
    source_triples = parse_triples(source_text)
    predicted_predicates = parse_predicted_predicates(pred_text)
    matched_source_indices = set()
    matched_predicate_indices = set()

    for pred_idx, predicted in enumerate(predicted_predicates):
        for source_idx, triple in enumerate(source_triples):
            if source_idx in matched_source_indices:
                continue
            if predicted == triple["predicate"]:
                matched_source_indices.add(source_idx)
                matched_predicate_indices.add(pred_idx)
                break

    omitted = [triple for idx, triple in enumerate(source_triples) if idx not in matched_source_indices]
    added = [
        {"predicted_predicate_index": idx, "predicate": predicate}
        for idx, predicate in enumerate(predicted_predicates)
        if idx not in matched_predicate_indices
    ]

    return source_triples, predicted_predicates, matched_source_indices, matched_predicate_indices, omitted, added


def read_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def result_path(task, dataset):
    suffix = "ordering" if task == "ordering" else "structuring"
    return RESULTS_DIR / task / f"{dataset}_{suffix}.json"


def display_rows(rows, limit=20):
    rows = list(rows)
    if pd is not None:
        return pd.DataFrame(rows).head(limit)
    return rows[:limit]


def print_table(rows, columns=None, limit=25, title=None, max_width=42):
    """Print a compact table so notebook output is readable even without pandas."""
    rows = list(rows)[:limit]
    if title:
        print(f"\n{title}")
        print("=" * len(title))
    if not rows:
        print("No rows.")
        return

    if columns is None:
        columns = list(rows[0].keys())

    if pd is not None:
        print(pd.DataFrame(rows)[columns].to_string(index=False))
        return

    def cell(value):
        value = str(value)
        value = value.replace("\n", " ")
        return value if len(value) <= max_width else value[: max_width - 3] + "..."

    widths = {
        column: min(max(len(str(column)), *(len(cell(row.get(column, ""))) for row in rows)), max_width)
        for column in columns
    }
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(separator)
    for row in rows:
        print(" | ".join(cell(row.get(column, "")).ljust(widths[column]) for column in columns))


def write_csv(path, rows):
    rows = list(rows)
    fieldnames = sorted({key for row in rows for key in row.keys()})
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def make_summary(rows, group_keys):
    """
    Aggregate detail rows into summary statistics, grouped by group_keys.

    For each group the following counts are produced:
      num_examples                — number of triple sets (examples) in the group
      total_source_triples        — total source triples across all examples
      total_predicted_predicates  — total predicted predicates across all examples
      total_matched_source_triples— source triples that were successfully matched
      total_omitted_triples       — source triples with no matching prediction
      total_added_triples         — predicted predicates with no matching source triple
      examples_with_omissions     — how many examples had at least one omission
      examples_with_additions     — how many examples had at least one addition
      omitted_triples_pct         — total_omitted / total_source * 100
      added_triples_pct_vs_source — total_added  / total_source * 100
                                    (denominator is source triples, not predicted)
      examples_with_omissions_pct — examples_with_omissions / num_examples * 100
      examples_with_additions_pct — examples_with_additions / num_examples * 100
    """
    grouped = defaultdict(list)
    for row in rows:
        grouped[tuple(row[key] for key in group_keys)].append(row)

    summary = []
    for key_values, group in sorted(grouped.items()):
        row = dict(zip(group_keys, key_values))
        row["num_examples"] = len(group)
        row["total_source_triples"] = sum(item["num_source_triples"] for item in group)
        row["total_predicted_predicates"] = sum(item["num_predicted_predicates"] for item in group)
        row["total_matched_source_triples"] = sum(item["num_matched_source_triples"] for item in group)
        row["total_omitted_triples"] = sum(item["num_omitted_triples"] for item in group)
        row["total_added_triples"] = sum(item["num_added_triples"] for item in group)
        row["examples_with_omissions"] = sum(item["num_omitted_triples"] > 0 for item in group)
        row["examples_with_additions"] = sum(item["num_added_triples"] > 0 for item in group)
        row["omitted_triples_pct"] = round(row["total_omitted_triples"] / row["total_source_triples"] * 100, 2) if row["total_source_triples"] else 0.0
        row["added_triples_pct_vs_source"] = round(row["total_added_triples"] / row["total_source_triples"] * 100, 2) if row["total_source_triples"] else 0.0
        row["examples_with_omissions_pct"] = round(row["examples_with_omissions"] / row["num_examples"] * 100, 2) if row["num_examples"] else 0.0
        row["examples_with_additions_pct"] = round(row["examples_with_additions"] / row["num_examples"] * 100, 2) if row["num_examples"] else 0.0
        summary.append(row)

    return summary


def predicate_frequency(rows, field_name):
    """
    Count how often each predicate appears in omitted_predicates or added_predicates,
    broken down by task, dataset, and comparison_basis.

    The field (e.g. omitted_predicates) stores predicates joined by ' | '.
    Each individual predicate is counted once per example row it appears in.
    """
    counts = defaultdict(int)
    for row in rows:
        value = str(row.get(field_name, ""))
        for predicate in value.split(" | "):
            if predicate:
                counts[(row["task"], row["dataset"], row["comparison_basis"], predicate)] += 1

    return sorted(
        [
            {
                "task": task,
                "dataset": dataset,
                "comparison_basis": comparison_basis,
                "predicate": predicate,
                "count": count,
            }
            for (task, dataset, comparison_basis, predicate), count in counts.items()
        ],
        key=lambda row: (row["task"], row["dataset"], row["comparison_basis"], -row["count"], row["predicate"]),
    )

print("Ready.")


Ready.


## Load original WebNLG-17 inputs from ordering results

In [74]:
webnlg17_inputs = {}

for dataset in DATASETS:
    path = result_path("ordering", dataset)
    records = read_json(path)
    webnlg17_inputs[dataset] = {
        record.get("idx", position): record.get("input", "")
        for position, record in enumerate(records)
    }

input_counts = {dataset: len(inputs) for dataset, inputs in webnlg17_inputs.items()}
print_table(
    [{"dataset": dataset, "num_webnlg17_inputs": count} for dataset, count in input_counts.items()],
    title="Loaded WebNLG-17 input sets",
)
input_counts


Loaded WebNLG-17 input sets
       dataset  num_webnlg17_inputs
       factual                 1779
     fictional                 1779
counterfactual                 1779


{'factual': 1779, 'fictional': 1779, 'counterfactual': 1779}

## Calculate omissions and additions

For every result file, `compare_predicates_to_source` matches each predicted predicate to a source triple using **greedy, case-sensitive matching**: the first unmatched source triple with an identical predicate string is consumed. Unmatched source triples become **omissions**; unmatched predicted predicates become **additions**.

### What `comparison_basis` means

The `comparison_basis` column records what is treated as the "true" source for a given row. This distinction is critical for interpreting the numbers:

| comparison_basis | task | What it measures |
|---|---|---|
| `webnlg17_input` | ordering | Triples lost by the **ordering** model relative to the original WebNLG-17 input. |
| `ordering_output` | structuring | Triples lost by the **structuring** model relative to what ordering gave it — i.e. errors introduced *by structuring alone*. |
| `webnlg17_input` | structuring | Triples lost across the **full pipeline** (ordering + structuring combined). Do **not** interpret this as structuring-specific error. |

For your thesis, the most meaningful numbers are:
- `webnlg17_input` / ordering → ordering-stage loss
- `ordering_output` / structuring → structuring-stage loss (isolated)
- `webnlg17_input` / structuring → total end-to-end loss


In [75]:
detail_rows = []
event_rows = []


def add_detail_and_events(task, dataset, comparison_basis, file_path, idx, position, source_text, pred_text):
    source_triples, predicted_predicates, matched_source, matched_predicates, omitted, added = compare_predicates_to_source(source_text, pred_text)
    triple_set_id = idx

    detail_row = {
        "task": task,
        "dataset": dataset,
        "comparison_basis": comparison_basis,
        "file": str(file_path.relative_to(PROJECT_DIR)),
        "triple_set_id": triple_set_id,
        "idx": idx,
        "position": position,
        "num_source_triples": len(source_triples),
        "num_predicted_predicates": len(predicted_predicates),
        "num_matched_source_triples": len(matched_source),
        "num_matched_predicates": len(matched_predicates),
        "num_omitted_triples": len(omitted),
        "num_added_triples": len(added),
        "omitted_predicates": " | ".join(triple["predicate"] for triple in omitted),
        "added_predicates": " | ".join(item["predicate"] for item in added),
        "omitted_triples": " || ".join(triple["raw"] for triple in omitted),
        "source_input": source_text,
        "pred": pred_text,
    }
    detail_rows.append(detail_row)

    for triple in omitted:
        event_rows.append({
            "change_type": "omitted",
            "task": task,
            "dataset": dataset,
            "comparison_basis": comparison_basis,
            "file": str(file_path.relative_to(PROJECT_DIR)),
            "triple_set_id": triple_set_id,
            "idx": idx,
            "position": position,
            "source_triple_index": triple["source_triple_index"],
            "predicted_predicate_index": "",
            "predicate": triple["predicate"],
            "subject": triple["subject"],
            "object": triple["object"],
            "triple": triple["raw"],
            "source_input": source_text,
            "pred": pred_text,
        })

    for item in added:
        event_rows.append({
            "change_type": "added",
            "task": task,
            "dataset": dataset,
            "comparison_basis": comparison_basis,
            "file": str(file_path.relative_to(PROJECT_DIR)),
            "triple_set_id": triple_set_id,
            "idx": idx,
            "position": position,
            "source_triple_index": "",
            "predicted_predicate_index": item["predicted_predicate_index"],
            "predicate": item["predicate"],
            "subject": "",
            "object": "",
            "triple": "",
            "source_input": source_text,
            "pred": pred_text,
        })


for dataset in DATASETS:
    ordering_path = result_path("ordering", dataset)
    ordering_records = read_json(ordering_path)
    for position, record in enumerate(ordering_records):
        idx = record.get("idx", position)
        add_detail_and_events(
            task="ordering",
            dataset=dataset,
            comparison_basis="webnlg17_input",
            file_path=ordering_path,
            idx=idx,
            position=position,
            source_text=record.get("input", ""),
            pred_text=record.get("pred", ""),
        )

    structuring_path = result_path("structuring", dataset)
    structuring_records = read_json(structuring_path)
    for position, record in enumerate(structuring_records):
        idx = record.get("idx", position)
        comparisons = [
            ("ordering_output", record.get("input", "")),
            ("webnlg17_input", webnlg17_inputs.get(dataset, {}).get(idx, "")),
        ]

        for comparison_basis, source_text in comparisons:
            add_detail_and_events(
                task="structuring",
                dataset=dataset,
                comparison_basis=comparison_basis,
                file_path=structuring_path,
                idx=idx,
                position=position,
                source_text=source_text,
                pred_text=record.get("pred", ""),
            )

summary_rows = make_summary(detail_rows, ["task", "dataset", "comparison_basis", "file"])
task_summary_rows = make_summary(detail_rows, ["task", "comparison_basis"])
changed_rows = [row for row in detail_rows if row["num_omitted_triples"] > 0 or row["num_added_triples"] > 0]

summary_columns = [
    "task", "dataset", "comparison_basis", "num_examples", "total_source_triples",
    "total_omitted_triples", "omitted_triples_pct", "total_added_triples",
    "added_triples_pct_vs_source", "examples_with_omissions", "examples_with_additions",
]
print_table(summary_rows, columns=summary_columns, limit=20, title="Dataset summary")
display_rows(summary_rows, limit=50)


Dataset summary
       task        dataset comparison_basis  num_examples  total_source_triples  total_omitted_triples  omitted_triples_pct  total_added_triples  added_triples_pct_vs_source  examples_with_omissions  examples_with_additions
   ordering counterfactual   webnlg17_input          1779                  5639                    639                11.33                  369                         6.54                      430                      263
   ordering        factual   webnlg17_input          1779                  5639                    697                12.36                  390                         6.92                      458                      270
   ordering      fictional   webnlg17_input          1779                  5639                    601                10.66                  321                         5.69                      401                      225
structuring counterfactual  ordering_output          1779                  5110        

,task,dataset,comparison_basis,file,num_examples,total_source_triples,total_predicted_predicates,total_matched_source_triples,total_omitted_triples,total_added_triples,examples_with_omissions,examples_with_additions,omitted_triples_pct,added_triples_pct_vs_source,examples_with_omissions_pct,examples_with_additions_pct
0,ordering,counterfactual,webnlg17_input,results/ordering/counterfactual_ordering.json,1779,5639,5369,5000,639,369,430,263,11.33,6.54,24.17,14.78
1,ordering,factual,webnlg17_input,results/ordering/factual_ordering.json,1779,5639,5332,4942,697,390,458,270,12.36,6.92,25.74,15.18
2,ordering,fictional,webnlg17_input,results/ordering/fictional_ordering.json,1779,5639,5359,5038,601,321,401,225,10.66,5.69,22.54,12.65
3,structuring,counterfactual,ordering_output,results/structuring/counterfactual_structuring...,1779,5110,5118,5104,6,14,6,14,0.12,0.27,0.34,0.79
4,structuring,counterfactual,webnlg17_input,results/structuring/counterfactual_structuring...,1779,5639,5118,5104,535,14,357,14,9.49,0.25,20.07,0.79
5,structuring,factual,ordering_output,results/structuring/factual_structuring.json,1779,5047,5049,5041,6,8,6,8,0.12,0.16,0.34,0.45
6,structuring,factual,webnlg17_input,results/structuring/factual_structuring.json,1779,5639,5049,5041,598,8,387,8,10.60,0.14,21.75,0.45
7,structuring,fictional,ordering_output,results/structuring/fictional_structuring.json,1779,5126,5122,5119,7,3,7,3,0.14,0.06,0.39,0.17
8,structuring,fictional,webnlg17_input,results/structuring/fictional_structuring.json,1779,5639,5122,5119,520,3,337,3,9.22,0.05,18.94,0.17


## Overall task summary

In [76]:
task_columns = [
    "task", "comparison_basis", "num_examples", "total_source_triples",
    "total_omitted_triples", "omitted_triples_pct", "total_added_triples",
    "added_triples_pct_vs_source", "examples_with_omissions", "examples_with_additions",
]
print_table(task_summary_rows, columns=task_columns, limit=20, title="Task summary")
display_rows(task_summary_rows, limit=20)


Task summary
       task comparison_basis  num_examples  total_source_triples  total_omitted_triples  omitted_triples_pct  total_added_triples  added_triples_pct_vs_source  examples_with_omissions  examples_with_additions
   ordering   webnlg17_input          5337                 16917                   1937                11.45                 1080                         6.38                     1289                      758
structuring  ordering_output          5337                 15283                     19                 0.12                   25                         0.16                       19                       25
structuring   webnlg17_input          5337                 16917                   1653                 9.77                   25                         0.15                     1081                       25


,task,comparison_basis,num_examples,total_source_triples,total_predicted_predicates,total_matched_source_triples,total_omitted_triples,total_added_triples,examples_with_omissions,examples_with_additions,omitted_triples_pct,added_triples_pct_vs_source,examples_with_omissions_pct,examples_with_additions_pct
0,ordering,webnlg17_input,5337,16917,16060,14980,1937,1080,1289,758,11.45,6.38,24.15,14.20
1,structuring,ordering_output,5337,15283,15289,15264,19,25,19,25,0.12,0.16,0.36,0.47
2,structuring,webnlg17_input,5337,16917,15289,15264,1653,25,1081,25,9.77,0.15,20.25,0.47


## Simple error statistics

This section calculates the headline error statistics and saves them to `results/omitted_triples/triple_error_statistics.csv`.

### Formulas

| Column | Formula | Plain-English meaning |
|---|---|---|
| `omitted_pct` | `total_omitted / total_source × 100` | What percentage of the expected triples did the model **omit**? |
| `added_pct` | `total_added / total_source × 100` | How many extra (hallucinated) predicates were generated, expressed as a **percentage of what was expected**? |
| `total_error_pct` | `(omitted + added) / total_source × 100` | Combined error rate (see warning below). |

### Important notes on the denominator

All three rates use **total_source_triples** as the denominator, not the number of predicted predicates. This means `added_pct` answers "how many additions occurred relative to the expected count?" — not "what fraction of the model's own predictions were wrong." If you need the latter, divide `total_added_triples` by `total_predicted_predicates` instead.

### ⚠ Do NOT report `total_error_pct` as a single headline figure in your thesis

`total_error_pct` sums omissions and additions together, but these are **opposite** types of error:
- An **omission** means the model *dropped* a triple (reduces coverage).
- An **addition** means the model *invented* a predicate that was not in the source (hallucination).

Adding them produces a number that obscures which problem is larger. Report `omitted_pct` and `added_pct` separately in your thesis. `total_error_pct` is kept in the CSV for completeness but should not be cited alone.


In [77]:
def add_error_percentages(rows):
    """
    Add headline error-rate columns to task or dataset summary rows.

    All rates use total_source_triples as the denominator so that omitted_pct
    and added_pct are directly comparable to each other.

    Columns added:
      omitted_pct     = total_omitted_triples / total_source_triples * 100
      added_pct       = total_added_triples   / total_source_triples * 100
      total_errors    = total_omitted_triples + total_added_triples
      total_error_pct = total_errors          / total_source_triples * 100

    WARNING: Do not report total_error_pct as a single headline number.
    Omissions (dropped triples) and additions (hallucinated predicates) are
    opposite error types. Report omitted_pct and added_pct separately.
    """
    stats = []
    for row in rows:
        total_source = row["total_source_triples"]
        omitted = row["total_omitted_triples"]
        added = row["total_added_triples"]
        total_errors = omitted + added

        omitted_pct = round(omitted / total_source * 100, 2) if total_source else 0.0
        added_pct = round(added / total_source * 100, 2) if total_source else 0.0
        total_error_pct = round(total_errors / total_source * 100, 2) if total_source else 0.0

        stats.append({
            **row,
            "total_errors": total_errors,
            "omitted_pct": omitted_pct,
            "added_pct": added_pct,
            "total_error_pct": total_error_pct,
        })
    return stats


simple_task_stats_rows = add_error_percentages(task_summary_rows)
simple_dataset_stats_rows = add_error_percentages(summary_rows)

# Columns to display and save for the task-level statistics CSV (thesis numbers live here)
simple_task_columns = [
    "task",
    "comparison_basis",
    "num_examples",
    "total_source_triples",
    "total_omitted_triples",
    "omitted_pct",               # → cite this in your thesis
    "total_added_triples",
    "added_pct",                 # → cite this in your thesis
    "total_errors",
    "total_error_pct",           # kept for completeness; do NOT cite alone
    "examples_with_omissions",
    "examples_with_omissions_pct",
    "examples_with_additions",
    "examples_with_additions_pct",
]

# Columns for the dataset-level breakdown (factual / fictional / counterfactual)
simple_dataset_columns = [
    "task",
    "dataset",
    "comparison_basis",
    "num_examples",
    "total_source_triples",
    "total_omitted_triples",
    "omitted_pct",
    "total_added_triples",
    "added_pct",
    "total_errors",
    "total_error_pct",
]

# NOTE: The statistics CSV is written once in the "Save results" cell below.
# A duplicate write that was here previously has been removed.

print_table(simple_task_stats_rows, columns=simple_task_columns, limit=20, title="Simple task-level error statistics")
print_table(simple_dataset_stats_rows, columns=simple_dataset_columns, limit=20, title="Simple dataset-level error statistics")

display_rows(simple_task_stats_rows, limit=20)



Simple task-level error statistics
       task comparison_basis  num_examples  total_source_triples  total_omitted_triples  omitted_pct  total_added_triples  added_pct  total_errors  total_error_pct  examples_with_omissions  examples_with_omissions_pct  examples_with_additions  examples_with_additions_pct
   ordering   webnlg17_input          5337                 16917                   1937        11.45                 1080       6.38          3017            17.83                     1289                        24.15                      758                        14.20
structuring  ordering_output          5337                 15283                     19         0.12                   25       0.16            44             0.29                       19                         0.36                       25                         0.47
structuring   webnlg17_input          5337                 16917                   1653         9.77                   25       0.15          1678  

,task,comparison_basis,num_examples,total_source_triples,total_predicted_predicates,total_matched_source_triples,total_omitted_triples,total_added_triples,examples_with_omissions,examples_with_additions,omitted_triples_pct,added_triples_pct_vs_source,examples_with_omissions_pct,examples_with_additions_pct,total_errors,omitted_pct,added_pct,total_error_pct
0,ordering,webnlg17_input,5337,16917,16060,14980,1937,1080,1289,758,11.45,6.38,24.15,14.20,3017,11.45,6.38,17.83
1,structuring,ordering_output,5337,15283,15289,15264,19,25,19,25,0.12,0.16,0.36,0.47,44,0.12,0.16,0.29
2,structuring,webnlg17_input,5337,16917,15289,15264,1653,25,1081,25,9.77,0.15,20.25,0.47,1678,9.77,0.15,9.92


## Structuring errors relative to ordering output

This section isolates errors made **by the structuring model alone**, by comparing its output against its direct input — the ordering model's output.

- `total_source_triples` here = the number of triples that ordering passed to structuring (not the original WebNLG-17 count).
- `omitted_pct` = percentage of ordering-output triples that structuring failed to reproduce.
- `added_pct` = extra predicates structuring invented beyond what ordering gave it, expressed as a percentage of the ordering-output triple count.

Because this comparison uses ordering's output as the baseline, these numbers **exclude** any triples already lost during ordering. This is the correct table to cite in your thesis when attributing errors specifically to the structuring stage.


In [78]:
# Filter to structuring task only, comparison_basis = ordering_output.
# This gives structuring-specific errors per dataset, independent of ordering-stage loss.
structuring_vs_ordering_rows = [
    row for row in simple_dataset_stats_rows
    if row["task"] == "structuring" and row["comparison_basis"] == "ordering_output"
]

structuring_vs_ordering_columns = [
    "dataset",
    "num_examples",
    "total_source_triples",       # = ordering-output triples fed into structuring
    "total_omitted_triples",      # triples ordering gave structuring but structuring dropped
    "omitted_pct",                # omitted / ordering_output_triples * 100  ← cite this
    "total_added_triples",        # predicates structuring invented beyond ordering output
    "added_pct",                  # added   / ordering_output_triples * 100  ← cite this
    "examples_with_omissions",
    "examples_with_omissions_pct",
    "examples_with_additions",
    "examples_with_additions_pct",
]

print_table(
    structuring_vs_ordering_rows,
    columns=structuring_vs_ordering_columns,
    limit=10,
    title="Structuring errors relative to ordering output (structuring-specific errors only)",
)
display_rows(structuring_vs_ordering_rows, limit=10)



Structuring errors relative to ordering output (structuring-specific errors only)
       dataset  num_examples  total_source_triples  total_omitted_triples  omitted_pct  total_added_triples  added_pct  examples_with_omissions  examples_with_omissions_pct  examples_with_additions  examples_with_additions_pct
counterfactual          1779                  5110                      6         0.12                   14       0.27                        6                         0.34                       14                         0.79
       factual          1779                  5047                      6         0.12                    8       0.16                        6                         0.34                        8                         0.45
     fictional          1779                  5126                      7         0.14                    3       0.06                        7                         0.39                        3                         0.17


,task,dataset,comparison_basis,file,num_examples,total_source_triples,total_predicted_predicates,total_matched_source_triples,total_omitted_triples,total_added_triples,examples_with_omissions,examples_with_additions,omitted_triples_pct,added_triples_pct_vs_source,examples_with_omissions_pct,examples_with_additions_pct,total_errors,omitted_pct,added_pct,total_error_pct
0,structuring,counterfactual,ordering_output,results/structuring/counterfactual_structuring...,1779,5110,5118,5104,6,14,6,14,0.12,0.27,0.34,0.79,20,0.12,0.27,0.39
1,structuring,factual,ordering_output,results/structuring/factual_structuring.json,1779,5047,5049,5041,6,8,6,8,0.12,0.16,0.34,0.45,14,0.12,0.16,0.28
2,structuring,fictional,ordering_output,results/structuring/fictional_structuring.json,1779,5126,5122,5119,7,3,7,3,0.14,0.06,0.39,0.17,10,0.14,0.06,0.20


## Thesis summary tables

Three tables for reporting in the thesis:

- **Table 1** — Ordering errors per dataset (factual / fictional / counterfactual), measured against the original WebNLG-17 input.
- **Table 2** — Structuring-specific errors per dataset, measured against the ordering output (i.e. what the structuring model actually received as input). This isolates the structuring stage.
- **Table 3** — Cumulative pipeline loss per dataset, measured against the original WebNLG-17 input. Reflects combined loss from both ordering and structuring. Do not attribute to structuring alone.

Each table includes a TOTAL row computed from the per-dataset rows.


In [79]:
def build_thesis_table(stats_rows, task, comparison_basis, title):
    """
    Build one thesis summary table and print it.

    Steps:
      1. Filter simple_dataset_stats_rows to the requested task and comparison_basis.
      2. Select only the columns needed for the table and rename them for readability.
      3. Compute a TOTAL row by summing counts and recalculating percentages from
         those sums — NOT by averaging the per-dataset percentages, which would be
         wrong because each dataset has the same number of examples.
      4. Append the TOTAL row and display.

    Why recompute percentages for the total row?
      Per-dataset percentages are already correct for each dataset. For the total
      row we need: total_omitted / total_source * 100 across all three datasets
      combined. Averaging 10.60, 10.66, 11.33 would give the same answer here
      (because all datasets are the same size), but computing from raw counts is
      always correct regardless of dataset size balance.
    """
    # --- Step 1: filter to the requested task and comparison ---
    subset = [
        r for r in stats_rows
        if r["task"] == task and r["comparison_basis"] == comparison_basis
    ]

    # --- Step 2: compute the TOTAL row from raw counts ---
    total_source    = sum(r["total_source_triples"]   for r in subset)
    total_omitted   = sum(r["total_omitted_triples"]  for r in subset)
    total_added     = sum(r["total_added_triples"]    for r in subset)
    total_examples  = sum(r["num_examples"]           for r in subset)
    total_w_omit    = sum(r["examples_with_omissions"]for r in subset)
    total_w_add     = sum(r["examples_with_additions"]for r in subset)

    totals_row = {
        "dataset":                    "TOTAL",
        "num_examples":               total_examples,
        "total_source_triples":       total_source,
        "total_omitted_triples":      total_omitted,
        "omitted_pct":                round(total_omitted / total_source * 100, 2) if total_source else 0.0,
        "total_added_triples":        total_added,
        "added_pct":                  round(total_added   / total_source * 100, 2) if total_source else 0.0,
        "examples_with_omissions":    total_w_omit,
        "examples_with_omissions_pct":round(total_w_omit  / total_examples * 100, 2) if total_examples else 0.0,
        "examples_with_additions":    total_w_add,
        "examples_with_additions_pct":round(total_w_add   / total_examples * 100, 2) if total_examples else 0.0,
    }

    # --- Step 3: combine per-dataset rows + total row ---
    all_rows = subset + [totals_row]

    # --- Step 4: select and rename columns for display ---
    col_map = {
        "dataset":                     "Dataset",
        "num_examples":                "Examples",
        "total_source_triples":        "Source Triples",
        "total_omitted_triples":       "Omitted",
        "omitted_pct":                 "Omission %",
        "total_added_triples":         "Added",
        "added_pct":                   "Addition %",
        "examples_with_omissions":     "w/ Omissions",
        "examples_with_omissions_pct": "Omissions %",
        "examples_with_additions":     "w/ Additions",
        "examples_with_additions_pct": "Additions %",
    }

    df = (
        pd.DataFrame(all_rows)[list(col_map.keys())]
        .rename(columns=col_map)
    )

    print(f"\n{title}")
    print("=" * len(title))
    print(df.to_string(index=False))
    return df


# ------------------------------------------------------------------
# Table 1: Ordering — how many triples were lost or added relative
#          to the original WebNLG-17 input.
# ------------------------------------------------------------------
table1 = build_thesis_table(
    simple_dataset_stats_rows,
    task="ordering",
    comparison_basis="webnlg17_input",
    title="Table 1: Ordering — omission and addition rates by dataset (vs WebNLG-17 input)",
)

# ------------------------------------------------------------------
# Table 2: Structuring — errors introduced by the structuring model
#          ONLY, measured against what ordering gave it.
#          Source Triples here = ordering-output triple count.
# ------------------------------------------------------------------
table2 = build_thesis_table(
    simple_dataset_stats_rows,
    task="structuring",
    comparison_basis="ordering_output",
    title="Table 2: Structuring — structuring-specific errors by dataset (vs ordering output)",
)

# ------------------------------------------------------------------
# Table 3: Structuring — cumulative pipeline loss measured against
#          the original WebNLG-17 input. Includes triples already
#          lost during ordering. Do NOT attribute to structuring alone.
# ------------------------------------------------------------------
table3 = build_thesis_table(
    simple_dataset_stats_rows,
    task="structuring",
    comparison_basis="webnlg17_input",
    title="Table 3: Structuring — cumulative pipeline loss by dataset (vs WebNLG-17 input)",
)



Table 1: Ordering — omission and addition rates by dataset (vs WebNLG-17 input)
       Dataset  Examples  Source Triples  Omitted  Omission %  Added  Addition %  w/ Omissions  Omissions %  w/ Additions  Additions %
counterfactual      1779            5639      639       11.33    369        6.54           430        24.17           263        14.78
       factual      1779            5639      697       12.36    390        6.92           458        25.74           270        15.18
     fictional      1779            5639      601       10.66    321        5.69           401        22.54           225        12.65
         TOTAL      5337           16917     1937       11.45   1080        6.38          1289        24.15           758        14.20

Table 2: Structuring — structuring-specific errors by dataset (vs ordering output)
       Dataset  Examples  Source Triples  Omitted  Omission %  Added  Addition %  w/ Omissions  Omissions %  w/ Additions  Additions %
counterfactual      1779 

## Triple sets with omissions or additions

In [80]:
preview_columns = [
    "task",
    "dataset",
    "comparison_basis",
    "triple_set_id",
    "num_source_triples",
    "num_matched_source_triples",
    "num_omitted_triples",
    "num_added_triples",
    "omitted_predicates",
    "added_predicates",
]

print_table(changed_rows, columns=preview_columns, limit=50, title="Triple sets with omissions or additions")
display_rows([{key: row[key] for key in preview_columns} for row in changed_rows], limit=50)


Triple sets with omissions or additions
    task dataset comparison_basis  triple_set_id  num_source_triples  num_matched_source_triples  num_omitted_triples  num_added_triples                                    omitted_predicates                        added_predicates
ordering factual   webnlg17_input              4                   5                           4                    1                  1                                                  type                                    city
ordering factual   webnlg17_input             16                   4                           3                    1                  0                                              industry                                        
ordering factual   webnlg17_input             20                   7                           2                    5                  2 artist | producer | runtime | recordedIn | recordedIn                     recordIn | location
ordering factual   webnlg17_input  

,task,dataset,comparison_basis,triple_set_id,num_source_triples,num_matched_source_triples,num_omitted_triples,num_added_triples,omitted_predicates,added_predicates
0,ordering,factual,webnlg17_input,4,5,4,1,1,type,city
1,ordering,factual,webnlg17_input,16,4,3,1,0,industry,
2,ordering,factual,webnlg17_input,20,7,2,5,2,artist | producer | runtime | recordedIn | rec...,recordIn | location
3,ordering,factual,webnlg17_input,23,3,2,1,1,producer,production_team
4,ordering,factual,webnlg17_input,26,3,2,1,1,writer,author
5,ordering,factual,webnlg17_input,27,3,2,1,0,professionalField,
6,ordering,factual,webnlg17_input,29,5,4,1,0,training,
7,ordering,factual,webnlg17_input,31,6,4,2,1,writer | writer,author
8,ordering,factual,webnlg17_input,32,3,2,1,1,writer,author
9,ordering,factual,webnlg17_input,35,5,4,1,0,producer,


## Individual omitted or added items with triple-set ids

In [81]:
event_columns = [
    "change_type",
    "task",
    "dataset",
    "comparison_basis",
    "triple_set_id",
    "source_triple_index",
    "predicted_predicate_index",
    "predicate",
    "subject",
    "object",
    "triple",
]

print_table(event_rows, columns=event_columns, limit=100, title="Individual omitted or added items")
display_rows([{key: row[key] for key in event_columns} for row in event_rows], limit=100)


Individual omitted or added items
change_type     task dataset comparison_basis  triple_set_id source_triple_index predicted_predicate_index         predicate                                  subject                                                                           object                                                                                                  triple
    omitted ordering factual   webnlg17_input              4                   2                                        type                             Ciudad_Ayala                                                                             City                                                                                  Ciudad_Ayala type City
      added ordering factual   webnlg17_input              4                                             2              city                                                                                                                                           

,change_type,task,dataset,comparison_basis,triple_set_id,source_triple_index,predicted_predicate_index,predicate,subject,object,triple
0,omitted,ordering,factual,webnlg17_input,4,2,,type,Ciudad_Ayala,City,Ciudad_Ayala type City
1,added,ordering,factual,webnlg17_input,4,,2,city,,,
2,omitted,ordering,factual,webnlg17_input,16,3,,industry,Hypermarcas,Pharmaceuticals,Hypermarcas industry Pharmaceuticals
3,omitted,ordering,factual,webnlg17_input,20,0,,artist,Bootleg_Series_Volume_1:_The_Quine_Tapes,The_Velvet_Underground,Bootleg_Series_Volume_1:_The_Quine_Tapes artis...
4,omitted,ordering,factual,webnlg17_input,20,1,,producer,Bootleg_Series_Volume_1:_The_Quine_Tapes,The_Velvet_Underground,Bootleg_Series_Volume_1:_The_Quine_Tapes produ...
...,...,...,...,...,...,...,...,...,...,...,...
95,added,ordering,factual,webnlg17_input,151,,2,creator,,,
96,omitted,ordering,factual,webnlg17_input,159,1,,writer,English_Without_Tears,Terence_Rattigan,English_Without_Tears writer Terence_Rattigan
97,omitted,ordering,factual,webnlg17_input,159,2,,runtime,English_Without_Tears,89.0,English_Without_Tears runtime 89.0
98,omitted,ordering,factual,webnlg17_input,159,3,,editing,English_Without_Tears,Alan_Jaggs,English_Without_Tears editing Alan_Jaggs


## Most commonly omitted predicates

In [82]:
omitted_predicate_summary_rows = predicate_frequency(changed_rows, "omitted_predicates")
print_table(omitted_predicate_summary_rows, limit=50, title="Most commonly omitted predicates")
display_rows(omitted_predicate_summary_rows, limit=50)


Most commonly omitted predicates
    task        dataset comparison_basis              predicate  count
ordering counterfactual   webnlg17_input               producer    107
ordering counterfactual   webnlg17_input                 writer     88
ordering counterfactual   webnlg17_input      professionalField     56
ordering counterfactual   webnlg17_input                runtime     41
ordering counterfactual   webnlg17_input                   type     37
ordering counterfactual   webnlg17_input                 artist     36
ordering counterfactual   webnlg17_input            citizenship     26
ordering counterfactual   webnlg17_input                editing     24
ordering counterfactual   webnlg17_input             recordedIn     21
ordering counterfactual   webnlg17_input               industry     18
ordering counterfactual   webnlg17_input                mission     18
ordering counterfactual   webnlg17_input icaoLocationIdentifier     13
ordering counterfactual   webnlg17_input   

,task,dataset,comparison_basis,predicate,count
0,ordering,counterfactual,webnlg17_input,producer,107
1,ordering,counterfactual,webnlg17_input,writer,88
2,ordering,counterfactual,webnlg17_input,professionalField,56
3,ordering,counterfactual,webnlg17_input,runtime,41
4,ordering,counterfactual,webnlg17_input,type,37
5,ordering,counterfactual,webnlg17_input,artist,36
6,ordering,counterfactual,webnlg17_input,citizenship,26
7,ordering,counterfactual,webnlg17_input,editing,24
8,ordering,counterfactual,webnlg17_input,recordedIn,21
9,ordering,counterfactual,webnlg17_input,industry,18


## Most commonly added predicates

In [83]:
added_predicate_summary_rows = predicate_frequency(changed_rows, "added_predicates")
print_table(added_predicate_summary_rows, limit=50, title="Most commonly added predicates")
display_rows(added_predicate_summary_rows, limit=50)


Most commonly added predicates
    task        dataset comparison_basis              predicate  count
ordering counterfactual   webnlg17_input                 author     51
ordering counterfactual   webnlg17_input             production     31
ordering counterfactual   webnlg17_input                creator     23
ordering counterfactual   webnlg17_input                 editor     23
ordering counterfactual   webnlg17_input                   time     21
ordering counterfactual   webnlg17_input        professionField     17
ordering counterfactual   webnlg17_input               recordIn     17
ordering counterfactual   webnlg17_input                  genre     12
ordering counterfactual   webnlg17_input ICAOLocationIdentifier     11
ordering counterfactual   webnlg17_input                country     10
ordering counterfactual   webnlg17_input                   city      9
ordering counterfactual   webnlg17_input            nationality      8
ordering counterfactual   webnlg17_input     

,task,dataset,comparison_basis,predicate,count
0,ordering,counterfactual,webnlg17_input,author,51
1,ordering,counterfactual,webnlg17_input,production,31
2,ordering,counterfactual,webnlg17_input,creator,23
3,ordering,counterfactual,webnlg17_input,editor,23
4,ordering,counterfactual,webnlg17_input,time,21
5,ordering,counterfactual,webnlg17_input,professionField,17
6,ordering,counterfactual,webnlg17_input,recordIn,17
7,ordering,counterfactual,webnlg17_input,genre,12
8,ordering,counterfactual,webnlg17_input,ICAOLocationIdentifier,11
9,ordering,counterfactual,webnlg17_input,country,10


## Save results

In [84]:
details_path = OUTPUT_DIR / "triple_omission_addition_details.csv"
changed_path = OUTPUT_DIR / "triple_omission_addition_examples_only.csv"
events_path = OUTPUT_DIR / "triple_omission_addition_events.csv"
summary_path = OUTPUT_DIR / "triple_omission_addition_summary.csv"
task_summary_path = OUTPUT_DIR / "triple_omission_addition_task_summary.csv"
omitted_predicates_path = OUTPUT_DIR / "omitted_predicates_summary.csv"
added_predicates_path = OUTPUT_DIR / "added_predicates_summary.csv"
statistics_path = OUTPUT_DIR / "triple_error_statistics.csv"
structuring_vs_ordering_path = OUTPUT_DIR / "structuring_errors_vs_ordering_output.csv"

write_csv(details_path, detail_rows)
write_csv(changed_path, changed_rows)
write_csv(events_path, event_rows)
write_csv(summary_path, summary_rows)
write_csv(task_summary_path, task_summary_rows)
write_csv(omitted_predicates_path, omitted_predicate_summary_rows)
write_csv(added_predicates_path, added_predicate_summary_rows)

# --- Build the combined statistics CSV ---
# Columns shared by both task-level and dataset-level rows.
stats_columns = [
    "task",
    "dataset",
    "comparison_basis",
    "num_examples",
    "total_source_triples",
    "total_omitted_triples",
    "omitted_pct",
    "total_added_triples",
    "added_pct",
    "total_errors",
    "total_error_pct",
    "examples_with_omissions",
    "examples_with_omissions_pct",
    "examples_with_additions",
    "examples_with_additions_pct",
]

# Task-level totals: dataset = "ALL" (aggregated across factual/fictional/counterfactual)
task_level_rows = [
    {**{col: row[col] for col in stats_columns if col != "dataset"}, "dataset": "ALL"}
    for row in simple_task_stats_rows
]

# Dataset-level rows: one row per (task, dataset, comparison_basis) combination
dataset_level_rows = [
    {col: row[col] for col in stats_columns}
    for row in simple_dataset_stats_rows
]

# Combine: dataset-level rows first, then the ALL totals
combined_stats_rows = dataset_level_rows + task_level_rows

write_csv(statistics_path, [{col: row[col] for col in stats_columns} for row in combined_stats_rows])

write_csv(
    structuring_vs_ordering_path,
    [{column: row[column] for column in structuring_vs_ordering_columns} for row in structuring_vs_ordering_rows],
)

print("Saved output CSV files:")
for path in [
    details_path,
    changed_path,
    events_path,
    summary_path,
    task_summary_path,
    omitted_predicates_path,
    added_predicates_path,
    statistics_path,
    structuring_vs_ordering_path,
]:
    print(f"  - {path.relative_to(PROJECT_DIR)}")

print(f"\nStatistics CSV contains {len(combined_stats_rows)} rows:")
print(f"  - {len(dataset_level_rows)} dataset-level rows (factual / fictional / counterfactual)")
print(f"  - {len(task_level_rows)} task-level total rows (dataset = ALL)")


Saved output CSV files:
  - results/omitted_triples/triple_omission_addition_details.csv
  - results/omitted_triples/triple_omission_addition_examples_only.csv
  - results/omitted_triples/triple_omission_addition_events.csv
  - results/omitted_triples/triple_omission_addition_summary.csv
  - results/omitted_triples/triple_omission_addition_task_summary.csv
  - results/omitted_triples/omitted_predicates_summary.csv
  - results/omitted_triples/added_predicates_summary.csv
  - results/omitted_triples/triple_error_statistics.csv
  - results/omitted_triples/structuring_errors_vs_ordering_output.csv

Statistics CSV contains 12 rows:
  - 9 dataset-level rows (factual / fictional / counterfactual)
  - 3 task-level total rows (dataset = ALL)


In [ ]:

# Column	Plain-English meaning
# task:	Which pipeline stage — ordering or structuring
# dataset:	Which dataset — factual, fictional, counterfactual, or ALL (the combined total)
# comparison_basis:	What the model output was compared against — webnlg17_input (original input) or ordering_output (what ordering gave to structuring)
# num_examples:	How many triple sets were evaluated (1,779 per dataset; 5,337 for ALL)
# total_source_triples:	Total number of triples in the source being compared against
# total_omitted_triples:	How many source triples the model failed to include in its output
# omitted_pct:	total_omitted_triples / total_source_triples × 100 — the omission rate
# total_added_triples:	How many extra predicates the model generated that were not in the source
# added_pct:	total_added_triples / total_source_triples × 100 — the addition rate
# total_errors:	total_omitted + total_added (kept for completeness — do not cite alone)
# total_error_pct:	total_errors / total_source_triples × 100 (do not cite alone in your thesis)
# examples_with_omissions:	Number of examples where at least one triple was omitted
# examples_with_omissions_pct:	examples_with_omissions / num_examples × 100
# examples_with_additions:	Number of examples where at least one extra predicate was generated
# examples_with_additions_pct:	examples_with_additions / num_examples × 100

# omitted_pct — your omission rate
# added_pct — your addition rate
# examples_with_omissions_pct — what proportion of examples were affected by omissions
# examples_with_additions_pct — what proportion of examples were affected by additions